In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('DOGE_raw.csv')

In [2]:
if "Unnamed: 0" in df.columns:
    df.drop(columns= ["Unnamed: 0"],inplace = True)
if not "Change" in df.columns:
    df['Change'] = (df['Close'] - df['Close'].shift(1))
if not "Change %" in df.columns:
    df['Change %'] = 100* df['Change']/ (df['Close'].shift(1))
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['year']  = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day']   = df['timestamp'].dt.day
df['hour']  = df['timestamp'].dt.hour
df['Date'] = df['timestamp'].dt.date
df['minute'] = df['timestamp'].dt.minute


In [3]:
df

,timestamp,Open,High,Low,Close,Vol,Change,Change %,year,month,day,hour,Date,minute
0,2025-01-16 01:10:00,0.37847,0.37931,0.37746,0.37872,3520484.0,NaN,NaN,2025,1,16,1,2025-01-16,10
1,2025-01-16 01:15:00,0.37871,0.37914,0.37691,0.37743,2905812.0,-0.00129,-0.340621,2025,1,16,1,2025-01-16,15
2,2025-01-16 01:20:00,0.37744,0.37744,0.37586,0.37700,4805063.0,-0.00043,-0.113928,2025,1,16,1,2025-01-16,20
3,2025-01-16 01:25:00,0.37700,0.37745,0.37652,0.37723,5452985.0,0.00023,0.061008,2025,1,16,1,2025-01-16,25
4,2025-01-16 01:30:00,0.37722,0.37722,0.37580,0.37617,4882456.0,-0.00106,-0.280996,2025,1,16,1,2025-01-16,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,2025-12-29 06:05:00,0.12667,0.12702,0.12667,0.12701,1740150.0,0.00035,0.276330,2025,12,29,6,2025-12-29,5
99996,2025-12-29 06:10:00,0.12701,0.12705,0.12700,0.12701,354946.0,0.00000,0.000000,2025,12,29,6,2025-12-29,10
99997,2025-12-29 06:15:00,0.12701,0.12722,0.12701,0.12716,1145983.0,0.00015,0.118101,2025,12,29,6,2025-12-29,15
99998,2025-12-29 06:20:00,0.12715,0.12718,0.12698,0.12702,1228990.0,-0.00014,-0.110098,2025,12,29,6,2025-12-29,20


In [4]:
def addRSI(df):
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -1 * delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()

    # 4. Calculate RS and RSI
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

df = addRSI(df)

In [5]:
def addMACD(df):
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    return df
df = addMACD(df)

In [6]:
def addBB(df):
    df['SMA_20'] = df['Close'].rolling(window=20).mean() # Middle Band
    std_dev = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['SMA_20'] + (2 * std_dev)
    df['BB_Lower'] = df['SMA_20'] - (2 * std_dev)
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
    return df
df = addBB(df)

In [7]:
def addATR(df):
    high_low = df['High'] - df['Low']
    high_prev = (df['High'] - df['Close'].shift(1)).abs()
    low_prev = (df['Low'] - df['Close'].shift(1)).abs()
    df['TR'] = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1)
    df['ATR'] = df['TR'].rolling(window=14).mean()
    return df
df = addATR(df)

In [8]:
def addStochOsc(df):
    low_14 = df['Low'].rolling(14).min()
    high_14 = df['High'].rolling(14).max()
    df['Stoch_%K'] = 100 * (df['Close'] - low_14) / (high_14 - low_14)
    df['Stoch_%D'] = df['Stoch_%K'].rolling(3).mean()
    return df
df = addStochOsc(df)

In [9]:
df.dropna(inplace=True)

In [10]:
import pandas as pd
import numpy as np

def prepare_financial_features(df):
    """
    Transforms raw OHLCV data into stationary features for
    CNN-LSTM-VAE models.
    """

    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['log_range'] = np.log(df['High'] / df['Low'])
    df['rel_body'] = (df['Close'] - df['Open']) / (df['High'] - df['Low'] + 1e-8)
    df['log_vol_change'] = np.log(df['Vol'] / df['Vol'].shift(1) + 1e-8)
    df['rsi_norm'] = df['RSI'] / 100.0
    df['macd_rel'] = df['MACD'] / df['Close']

    # minute (0-59)
    df['min_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['min_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

    # Hour (0-23)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Day of WEEK (0-6) — fixed: previous version divided day-of-month by 7
    dow = pd.to_datetime(df['timestamp']).dt.dayofweek
    df['day_sin'] = np.sin(2 * np.pi * dow / 7)
    df['day_cos'] = np.cos(2 * np.pi * dow / 7)

    # Month (1-12)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df.drop(columns = ['timestamp','Date'],inplace=True)
    df = df.dropna().reset_index(drop=True)
    return df

df = prepare_financial_features(df)
if "Unnamed: 0" in df.columns:
    df.drop(columns = ['Unnamed: 0'],inplace=True)


In [11]:
df

,Open,High,Low,Close,Vol,Change,Change %,year,month,day,...,rsi_norm,macd_rel,min_sin,min_cos,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,0.37611,0.37660,0.37544,0.37624,5371317.0,0.00014,0.037224,2025,1,16,...,0.261585,-0.001193,-0.866025,5.000000e-01,0.500000,8.660254e-01,0.433884,-0.900969,5.000000e-01,0.866025
1,0.37623,0.37737,0.37570,0.37718,4607563.0,0.00094,0.249841,2025,1,16,...,0.320545,-0.001090,-0.500000,8.660254e-01,0.500000,8.660254e-01,0.433884,-0.900969,5.000000e-01,0.866025
2,0.37719,0.37822,0.37716,0.37766,4606096.0,0.00048,0.127260,2025,1,16,...,0.349124,-0.000897,0.000000,1.000000e+00,0.707107,7.071068e-01,0.433884,-0.900969,5.000000e-01,0.866025
3,0.37765,0.37798,0.37724,0.37795,3368453.0,0.00029,0.076789,2025,1,16,...,0.366462,-0.000675,0.500000,8.660254e-01,0.707107,7.071068e-01,0.433884,-0.900969,5.000000e-01,0.866025
4,0.37796,0.37857,0.37761,0.37787,7380531.0,-0.00008,-0.021167,2025,1,16,...,0.363585,-0.000510,0.866025,5.000000e-01,0.707107,7.071068e-01,0.433884,-0.900969,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99975,0.12667,0.12702,0.12667,0.12701,1740150.0,0.00035,0.276330,2025,12,29,...,0.467689,-0.000281,0.500000,8.660254e-01,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000
99976,0.12701,0.12705,0.12700,0.12701,354946.0,0.00000,0.000000,2025,12,29,...,0.467689,-0.000322,0.866025,5.000000e-01,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000
99977,0.12701,0.12722,0.12701,0.12716,1145983.0,0.00015,0.118101,2025,12,29,...,0.508151,-0.000257,1.000000,2.832769e-16,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000
99978,0.12715,0.12718,0.12698,0.12702,1228990.0,-0.00014,-0.110098,2025,12,29,...,0.472083,-0.000291,0.866025,-5.000000e-01,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000


In [12]:
df.to_csv("DOGE.csv")